In [1]:
# Cell 1: import required library
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_parquet("../data/processed/ca1_long.parquet")
print("Shape:", df.shape)

Shape: (5918109, 19)


In [3]:
# Cell 2: seupt and looks
cols = ["item_id", "dept_id", "cat_id", "sales", "d_num", "date", "sell_price"]
H = 28
LAST_DAY = int(df["d_num"].max())
TEST_START = LAST_DAY - H + 1

first_day = (df.loc[df["sell_price"].notna()]
             .groupby("item_id", observed=True)["d_num"].min().rename("first_day"))
df = df.merge(first_day, on="item_id", how="left")
df = df[df["d_num"] >= df["first_day"]].reset_index(drop=True)
df["sales"] = df["sales"].astype("float32")

Y = df.pivot(index="item_id", columns="d_num", values="sales").reindex(columns=range(1, LAST_DAY + 1))
Y_np = Y.to_numpy()
date_of = df.drop_duplicates("d_num").set_index("d_num")["date"]

# folds 
rows = []
for i in (1, 2, 3):
    val_end = TEST_START - 1 -(3 - i) * H
    val_start = val_end - H + 1
    rows.append({"fold": i, "train_end": val_start - 1, "val_start": val_start, "val_end": val_end})
folds = pd.DataFrame(rows)

# metrics
def _valid(y, p):
    m = ~np.isnan(y) & ~np.isnan(p)
    return y[m], p[m]

def mae(y, p):
    y, p = _valid(y, p)
    return float(np.abs(y - p).mean())

def rmse(y, p):
    y, p = _valid(y, p)
    return float(np.sqrt(((y - p) ** 2).mean()))

def wape(y, p):
    y, p = _valid(y, p)
    return float(np.abs(y - p).sum() / y.sum())

def mase_scale(Y_train, m = 7):
    d = np.abs(Y_train[:, m:] - Y_train[:, :-m])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category = RuntimeWarning)
        return np.nanmean(d, axis=1)

def mase(y, p, scale):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        ratio = np.nanmean(np.abs(y - p), axis = 1) / scale
    ok = np.isfinite(ratio) & (scale > 0)
    return float(ratio[ok].mean()), int((~ok).sum())

def score(Y_val, P, Y_train):
    m, skipped = mase(Y_val, P, mase_scale(Y_train))
    return {"MAE": mae(Y_val, P), "RMSE": rmse(Y_val, P),
            "WAPE": wape(Y_val, P), "MASE": m, "mase_items_skipped": skipped}

# baselines
def _finish(fc):
    return np.nan_to_num(fc, nan=0.0)

def zeros(Y_train, h=H):
    return np.zeros((Y_train.shape[0], h), dtype="float32")

def naive(Y_train, h=H):
    return _finish(np.repeat(Y_train[:, -1][:, None], h, axis=1))


def seasonal_naive(Y_train, h=H, m=7):
    return _finish(np.tile(Y_train[:, -m:], (1, int(np.ceil(h / m))))[:, :h])

def moving_average(Y_train, h=H, w=28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        ma = np.nanmean(Y_train[:, -w:], axis=1)
    return _finish(np.repeat(ma[:, None], h, axis=1))

# bactest
def backtest(forecaster, name, Y_all, folds, h=H, **kw):
    out = []
    for _, f in folds.iterrows():
        te, vs, ve = int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        assert vs == te + 1 and ve - vs + 1 == h
        Y_train, Y_val = Y_all[:, :te], Y_all[:, vs -1: ve]
        P = forecaster(Y_train, h=h, **kw)
        assert P.shape == Y_val.shape
        out.append({"model": name, "fold": int(f["fold"]), **score(Y_val, P, Y_train)})
    return pd.DataFrame(out)

# re-check 
print("Y shape:", Y.shape)
check = backtest(moving_average, "moving average (28d)", Y_np, folds)
print("moving average WAPE (all items):", round(check["WAPE"].mean(), 3))

Y shape: (3049, 1941)
moving average WAPE (all items): 0.748


In [5]:
#Cell 3: sample of 300 items, 100 per category, on sale since day 1000 or earlier
meta = (df.drop_duplicates("item_id")[["item_id", "cat_id"]]
        .merge(first_day.reset_index(), on="item_id"))
meta["item_id"] = meta["item_id"].astype(str)
meta["cat_id"] = meta["cat_id"].astype(str)

eligible = meta[meta["first_day"] <= 1000]
print("eligible items:", len(eligible), "of", len(meta))
print(eligible["cat_id"].value_counts().to_string())

parts = [g.sample(n=min(100, len(g)), random_state=42) for _, g in eligible.groupby("cat_id")]
sample = pd.concat(parts).reset_index(drop=True)
sample_ids = sample["item_id"].tolist()

print("\nsample size:", len(sample_ids))
print(sample["cat_id"].value_counts().to_string())

eligible items: 2601 of 3049
cat_id
FOODS        1221
HOUSEHOLD     881
HOBBIES       499

sample size: 300
cat_id
FOODS        100
HOBBIES      100
HOUSEHOLD    100


In [7]:
# Cell 4: convert to statsforecast format (only days up to d_1913)
TRAIN_END = TEST_START - 1
dates = pd.to_datetime(date_of.reindex(range(1, LAST_DAY + 1)).values)

pos = pd.Series(np.arange(len(Y)), index=Y.index.astype(str))
rows_idx = pos[sample_ids].to_numpy()

Ys = Y_np[rows_idx, :TRAIN_END]
sf_df = pd.DataFrame({
    "unique_id": np.repeat(sample_ids, TRAIN_END),
    "ds": np.tile(dates[:TRAIN_END], len(sample_ids)),
    "y": Ys.ravel().astype("float64"),
}).dropna(subset=["y"]).reset_index(drop=True)


g = sf_df.groupby("unique_id")["ds"].agg(["min", "max", "count"])
g["expected"] = (g["max"] - g["min"]).dt.days + 1
print("series:", len(g))
print("all gap-free:", (g["count"] == g["expected"]).all())
print("last date of every series:", g["max"].unique())
print("rows:", len(sf_df))
display(sf_df.head())

series: 300
all gap-free: True
last date of every series: <DatetimeArray>
['2016-04-24 00:00:00']
Length: 1, dtype: datetime64[ns]
rows: 492686


,unique_id,ds,y
0,FOODS_3_022,2013-09-07,0.0
1,FOODS_3_022,2013-09-08,0.0
2,FOODS_3_022,2013-09-09,0.0
3,FOODS_3_022,2013-09-10,1.0
4,FOODS_3_022,2013-09-11,0.0


In [9]:
# Cell 5: Time test on 5 items
import time
from statsforecast import StatsForecast
from statsforecast.models import (SeasonalNaive, WindowAverage, CrostonOptimized,
                                  ADIDA, IMAPA, AutoETS, AutoARIMA)

test_ids = sample_ids[::60]            # 5 items from the sample
sf_test = sf_df[sf_df["unique_id"].isin(test_ids)]
print("test items:", test_ids)

fast = [SeasonalNaive(season_length=7), WindowAverage(window_size=28),
        CrostonOptimized(), ADIDA(), IMAPA()]
ets = [AutoETS(season_length=7)]
arima = [AutoARIMA(season_length=7)]

def timed(models, label):
    t0 = time.time()
    sf = StatsForecast(models=models, freq="D", n_jobs=1)
    cv = sf.cross_validation(df=sf_test, h=H, step_size=H, n_windows=3)
    sec = time.time() - t0
    print(f"{label}: {sec:.1f}s for {len(test_ids)} items"
          f"-> about {sec / len(test_ids) * 300 / 60:.1f} min for 300 items")
    return cv

cv_fast = timed(fast, "fast models")
cv_ets = timed(ets, "AutoETS")
cv_arima = timed(arima, "AutoARIMA")

print("\nrows:", len(cv_fast), "(expect 420 = 5 items x 28 days x 3 windows)")
print("cutoffs:", [str(c.date()) for c in sorted(cv_fast["cutoff"].unique())])
display(cv_fast.head())

test items: ['FOODS_3_022', 'FOODS_3_685', 'HOBBIES_1_306', 'HOBBIES_1_366', 'HOUSEHOLD_1_166']
fast models: 0.0s for 5 items-> about 0.0 min for 300 items
AutoETS: 2.9s for 5 items-> about 2.9 min for 300 items
AutoARIMA: 118.6s for 5 items-> about 118.6 min for 300 items

rows: 420 (expect 420 = 5 items x 28 days x 3 windows)
cutoffs: ['2016-01-31', '2016-02-28', '2016-03-27']


,unique_id,ds,cutoff,y,SeasonalNaive,WindowAverage,CrostonOptimized,ADIDA,IMAPA
0,FOODS_3_022,2016-02-01,2016-01-31,1.0,0.0,0.464286,0.368481,0.357097,0.35198
1,FOODS_3_022,2016-02-02,2016-01-31,1.0,0.0,0.464286,0.368481,0.357097,0.35198
2,FOODS_3_022,2016-02-03,2016-01-31,0.0,0.0,0.464286,0.368481,0.357097,0.35198
3,FOODS_3_022,2016-02-04,2016-01-31,0.0,1.0,0.464286,0.368481,0.357097,0.35198
4,FOODS_3_022,2016-02-05,2016-01-31,0.0,0.0,0.464286,0.368481,0.357097,0.35198


In [13]:
# Cell 6: fit all models except AutoARIMA on 300 items, then score
models = [SeasonalNaive(season_length=7), WindowAverage(window_size=28),
          CrostonOptimized(), ADIDA(), IMAPA(), AutoETS(season_length=7)]

t0 = time.time()
sf = StatsForecast(models=models, freq="D", n_jobs=1)
cv_all = sf.cross_validation(df=sf_df, h=H, step_size=H, n_windows=3)
print(f"Fit time:, {time.time() - t0:.0f}s | rows: {len(cv_all)} (expected 25200)")

# scoring (same metric functions as the baselines)
pos = pd.Series(np.arange(len(Y)), index=Y.index.astype(str))
rows_idx = pos[sample_ids].to_numpy()

def score_cv(cv, ids):
    names = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y")]
    cutoffs = sorted(cv["cutoff"].unique())
    out = []
    for (_, f), cutoff in zip(folds.iterrows(), cutoffs):
        te, vs, ve = int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        assert pd.Timestamp(cutoff) == pd.Timestamp(date_of[te]), "cutoff does not match fold"
        sub = cv[cv["cutoff"] == cutoff]
        Y_train, Y_val = Y_np[rows_idx, :te], Y_np[rows_idx, vs - 1: ve]
        y_cv = sub.pivot(index = "unique_id", columns = "ds", values="y").reindex(ids).to_numpy()
        assert np.allclose(y_cv, Y_val, equal_nan=True), "actual differs"
        for m in names:
            P = sub.pivot(index = "unique_id", columns = "ds", values = m).reindex(ids).to_numpy()
            out.append({"model": m, "fold": int(f["fold"]), **score(Y_val, P, Y_train)})
    return pd.DataFrame(out)

res_sf = score_cv(cv_all, sample_ids)

Y_sample = Y_np[rows_idx]
res_base = pd.concat([
    backtest(zeros, "zeros (reference)", Y_sample, folds),
    backtest(naive, "naive", Y_sample, folds),
    backtest(seasonal_naive, "seasonal naive ", Y_sample, folds),
    backtest(moving_average, "moving average", Y_sample, folds),
], ignore_index=True)

allres = pd.concat([res_base, res_sf], ignore_index=True)
summary = allres.groupby("model", sort=False)[["MAE", "RMSE", "WAPE", "MASE"]].mean().round(3)
display(summary.sort_values("WAPE"))
print("per fold WAPE:")
display(allres.pivot(index="model", columns="fold",values="WAPE").round(3))

Fit time:, 233s | rows: 25200 (expected 25200)


,MAE,RMSE,WAPE,MASE
model,,,,
AutoETS,0.980,1.920,0.733,0.923
moving average,0.985,1.969,0.737,0.913
WindowAverage,0.985,1.969,0.737,0.913
ADIDA,1.001,1.999,0.749,0.926
IMAPA,1.002,2.002,0.749,0.925
CrostonOptimized,1.048,1.999,0.784,0.985
seasonal naive,1.156,2.458,0.865,1.041
SeasonalNaive,1.156,2.458,0.865,1.041
zeros (reference),1.336,3.458,1.000,0.911


per fold WAPE:


fold,1,2,3
model,,,
ADIDA,0.718,0.737,0.791
AutoETS,0.712,0.717,0.770
CrostonOptimized,0.764,0.766,0.820
IMAPA,0.716,0.739,0.793
SeasonalNaive,0.849,0.841,0.904
WindowAverage,0.708,0.729,0.774
moving average,0.708,0.729,0.774
naive,1.060,1.115,0.918
seasonal naive,0.849,0.841,0.904


In [ ]:
# Cell 7: AutoARIMA on 30 items

ids30 = sample.groupby("cat_id").head(10)["item_id"].tolist()
sf30 = sf_df[sf_df["unique_id"].isin(ids30)]
print("items:", len(ids30), "| expect", len(ids30) * 28 * 3, "forecast rows")

t0 = time.time()
sf_m = StatsForecast(models=[AutoARIMA(season_length=7), AutoETS(season_length=7),
                             WindowAverage(window_size=28)], freq="D", n_jobs=1)
cv30 = sf_m.cross_validation(df=sf30, h=H, step_size=H, n_windows=3)
print(f"fit time: {time.time() - t0:.0f}s | rows: {len(cv30)}")

def score_cv_ids(cv, ids):
    rows = pos[ids].to_numpy()
    names = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y")]
    cutoffs = sorted(cv["cutoff"].unique())
    out = []
    for (_, f), cutoff in zip(folds.iterrows(), cutoffs):
        te, vs, ve = int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        assert pd.Timestamp(cutoff) == pd.Timestamp(date_of[te]), "cutoff does not match fold"
        sub = cv[cv["cutoff"] == cutoff]
        Y_train, Y_val = Y_np[rows, :te], Y_np[rows, vs - 1: ve]
        y_cv = sub.pivot(index="unique_id", columns="ds", values="y").reindex(ids).to_numpy()
        assert np.allclose(y_cv, Y_val, equal_nan=True), "actuals differ"
        for m in names:
            P = sub.pivot(index="unique_id", columns="ds", values=m).reindex(ids).to_numpy()
            out.append({"model": m, "fold": int(f["fold"]), **score(Y_val, P, Y_train)})
    return pd.DataFrame(out)

res30 = score_cv_ids(cv30, ids30)
display(res30.groupby("model")[["MAE", "RMSE", "WAPE", "MASE"]].mean().round(3).sort_values("WAPE"))

items: 30 | expect 2520 forecast rows
fit time: 1307s | rows: 2520


,MAE,RMSE,WAPE,MASE
model,,,,
AutoETS,0.869,1.436,0.853,0.941
AutoARIMA,0.883,1.467,0.867,0.952
WindowAverage,0.888,1.497,0.872,0.954


## Summary of Notebook 04: statistical models

**Setup**
- Sample: 300 items (100 per category, seed 42), only items on sale since day 1000 or earlier (long history needed for weekly seasonality)
- Same 3 backtest folds as Notebook 03 (cutoffs 2016-01-31, 2016-02-28, 2016-03-27); statsforecast windows verified to match; final test window not in the data
- Same metric functions as the baselines; our baselines re-scored on the same 300 items

**Results (300 items, mean of 3 folds)**

| Model | MAE | RMSE | WAPE | MASE |
|---|---|---|---|---|
| AutoETS (season 7) | 0.980 | 1.920 | 0.733 | 0.923 |
| moving average (28d) | 0.985 | 1.969 | 0.737 | 0.913 |
| ADIDA | 1.001 | 1.999 | 0.749 | 0.926 |
| IMAPA | 1.002 | 2.002 | 0.749 | 0.925 |
| CrostonOptimized | 1.048 | 1.999 | 0.784 | 0.985 |
| seasonal naive (7d) | 1.156 | 2.458 | 0.865 | 1.041 |
| zeros (reference) | 1.336 | 3.458 | 1.000 | 0.911 |
| naive | 1.378 | 2.959 | 1.031 | 1.200 |

**AutoARIMA vs AutoETS vs moving average (30 items only, 44 s per item):** WAPE 0.867 / 0.853 / 0.872

**Findings**
- Classical models do not meaningfully beat a 28-day moving average (AutoETS leads by 0.004 WAPE, and loses in fold 1 and on MASE)
- Bar for LightGBM: about 0.73-0.75 WAPE at item-day level
- Fold 3 is harder for every model; the ranking stays the same
- Univariate models see only one item's history, so they cannot use events, SNAP, price or other items' weekday pattern

**Limits (for the README)**
- 300 long-history items; late launches not covered
- Default settings, no tuning
- AutoARIMA run on 30 items only (about 3.5 hours estimated for 300)
- Spring folds only (February-April)